# CharmMining — Demo Chương 1 → Chương 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheBrilliantCoder/frequent-itemset-mining/blob/main/notebooks/demo.ipynb)

Notebook tái lập **toàn bộ** dự án khai thác tập phổ biến đóng bằng thuật toán **CHARM** tự cài đặt bằng Julia:

| Phần | Nội dung |
|---|---|
| **0. Thiết lập** | Clone repo, cài Julia/Java, tải dữ liệu (kiểm tra SHA-256) |
| **1. Chương 1** | Chạy CHARM (bản cơ bản & tối ưu) trên CSDL mẫu, bộ kiểm thử tự động + đối chuẩn SPMF trên 5 benchmark |
| **2. Chương 2** | Thực nghiệm thời gian, số tập đóng, bộ nhớ và khả năng mở rộng: Basic vs. Optimized vs. SPMF |
| **3. Chương 3** | Ứng dụng 1 — Market Basket Analysis (UCI Online Retail); Ứng dụng 2 — Motif DNA (7 họ protein) |

Tất cả thực nghiệm chạy trên **toàn bộ** dữ liệu gốc (không lấy mẫu con). Cách chạy: **Runtime → Run all**.
Phần Chương 2 chạy lâu nhất vì mỗi điểm đo là một tiến trình riêng và chạy cả 3 cài đặt.

## 0. Thiết lập môi trường

Ô dưới đây clone repository (khi chạy trên Colab) và chuyển thư mục làm việc về gốc repo.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/TheBrilliantCoder/frequent-itemset-mining.git"
BRANCH = "fix/repo-cleanup"
JULIA_VERSION = "1.13.2"

def find_repo_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "src" / "CharmMining.jl").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    target = Path("/content/frequent-itemset-mining")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(target)], check=True)
    REPO_ROOT = target
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

In [ ]:
import shutil

def run(cmd, env=None):
    """Run a command, streaming its output into the notebook; fail the cell on error."""
    print("$", " ".join(cmd), flush=True)
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=full_env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")

# Julia (pinned version, same as local experiments)
if shutil.which("julia") is None:
    minor = ".".join(JULIA_VERSION.split(".")[:2])
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/{minor}/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    run(["bash", "-c", f"curl -fsSL --retry 3 {url} -o /tmp/julia.tar.gz && "
         "mkdir -p /opt/julia && tar -xzf /tmp/julia.tar.gz -C /opt/julia --strip-components 1 && "
         "ln -sf /opt/julia/bin/julia /usr/local/bin/julia"])

# Java 25+ (tools/spmf.jar is compiled for Java 25, class-file version 69).
# Colab's preinstalled Java is older, so install Temurin 25 and put it first on PATH;
# the Julia scripts call `java` from PATH.
JAVA_MIN_MAJOR = 25

def java_major():
    if shutil.which("java") is None:
        return 0
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    version = out.split('"')[1] if '"' in out else "0"
    return int(version.split(".")[0])

if java_major() < JAVA_MIN_MAJOR:
    jdk_dir = "/opt/temurin-25"
    url = "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jre/hotspot/normal/eclipse"
    run(["bash", "-c", f"curl -fsSL --retry 3 -L {url} -o /tmp/jre25.tar.gz && "
         f"mkdir -p {jdk_dir} && tar -xzf /tmp/jre25.tar.gz -C {jdk_dir} --strip-components 1"])
    os.environ["JAVA_HOME"] = jdk_dir
    os.environ["PATH"] = f"{jdk_dir}/bin:" + os.environ["PATH"]
assert java_major() >= JAVA_MIN_MAJOR, "Java 25+ is required by tools/spmf.jar"

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run(["julia", "--project=.", "-e", "using Pkg; Pkg.instantiate(); Pkg.precompile()"])
run(["julia", "--version"])
run(["java", "-version"])

### Tải dữ liệu

`scripts/download_data.py` tải 5 benchmark FIMI (`chess`, `mushrooms`, `retail`, `T10I4D100K`, `accidents`), `spmf.jar`
và tập UCI Online Retail từ kho dữ liệu Hugging Face của nhóm, rồi **kiểm tra SHA-256** của từng file để bảo đảm dùng đúng dữ liệu
như khi chạy cục bộ. Tập DNA (`data/applications/dna/human.txt`) có sẵn trong repository.

In [ ]:
run([sys.executable, "scripts/download_data.py"])

## 1. Chương 1 — Thuật toán CHARM

### 1.1. Chạy thử trên CSDL mẫu Pasquier 1999
`run.jl` khai thác `data/toy/contextPasquier99.txt` với `minsup = 2` bằng bản tối ưu `mine_file_bit`
và ghi kết quả theo chuẩn SPMF (`<items> #SUP: <count>`).

In [ ]:
run(["julia", "--project=.", "run.jl"])
print("\nInput (data/toy/contextPasquier99.txt):")
print(Path("data/toy/contextPasquier99.txt").read_text())
print("Closed itemsets (results/output.txt):")
print(Path("results/output.txt").read_text())

### 1.2. Bộ kiểm thử tự động & đối chuẩn SPMF
`test/runtests.jl` chạy unit test cho cả bản cơ bản (`mine_file`) và bản tối ưu (`mine_file_bit`), module sinh luật,
rồi đối chiếu kết quả với SPMF `Charm_bitset` trên CSDL mẫu và **toàn bộ 5 benchmark**.

In [ ]:
run(["julia", "--project=.", "test/runtests.jl"])

## 2. Chương 2 — Thực nghiệm & Đánh giá hiệu năng

Ba cài đặt được so sánh: **Basic** (`mine_file`, `BitSet`), **Optimized** (`mine_file_bit`, `BitVector` + `@simd`)
và **SPMF** `Charm_bitset` (Java). Mỗi lần chạy là một tiến trình riêng.

### 2.1. Thời gian chạy & số tập đóng theo `minsup`

In [ ]:
import pandas as pd
from IPython.display import SVG, display

run(["julia", "--project=.", "comparison/compare_time.jl"])
time_df = pd.read_csv("comparison/time_result.csv")
time_df["itemsets_match"] = (time_df.julia_itemsets == time_df.spmf_itemsets) & (time_df.optimized_itemsets == time_df.spmf_itemsets)
display(time_df)
print("Số điểm đo khớp số tập đóng với SPMF:", int(time_df.itemsets_match.sum()), "/", len(time_df))

In [ ]:
run(["julia", "--project=.", "comparison/plot_time.jl"])
for dataset in time_df.dataset.unique():
    print(f"=== {dataset} ===")
    display(SVG(filename=f"comparison/plots/{dataset}_runtime.svg"))
    display(SVG(filename=f"comparison/plots/{dataset}_itemsets.svg"))

### 2.2. Bộ nhớ đỉnh
Bộ nhớ đỉnh (RSS, `VmHWM`) của từng tiến trình; `net` là phần còn lại sau khi trừ chi phí khởi động Julia/JVM.

In [ ]:
run(["julia", "--project=.", "comparison/compare_memory.jl"])
display(pd.read_csv("comparison/memory_summary.csv"))
run(["julia", "--project=.", "comparison/plot_memory.jl"])
display(SVG(filename="comparison/plots/memory_peak.svg"))
display(SVG(filename="comparison/plots/memory_net.svg"))

### 2.3. Khả năng mở rộng
CSDL được chia thành các tập con lồng nhau 10%, 25%, 50%, 75%, 100% (giữ `minsup` tương đối), đo thời gian ròng và bộ nhớ.

In [ ]:
run(["julia", "--project=.", "comparison/compare_scalability.jl"])
display(pd.read_csv("comparison/scalability_summary.csv"))
display(pd.read_csv("comparison/scalability_analysis.csv"))
run(["julia", "--project=.", "comparison/plot_scalability.jl"])
for name in ["scalability_runtime", "scalability_loglog", "scalability_memory"]:
    display(SVG(filename=f"comparison/plots/{name}.svg"))

## 3. Chương 3 — Ứng dụng thực tế

### 3.1. Ứng dụng 1: Phân tích Giỏ hàng (Market Basket Analysis)

**Thuật toán sử dụng:** **Optimized CHARM** (`CharmMining.jl`, tự cài đặt từ đầu bằng Julia — Chương 1) kết hợp bộ sinh luật kết hợp dựa trên **bao đóng Galois (`ClosureSupportIndex`)** (`src/rules.jl`).

**Tập dữ liệu:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) (giao dịch bán lẻ trực tuyến tại Vương quốc Anh, 12/2010 – 12/2011, 541.909 dòng hóa đơn).

### Thiết kế biến đổi dữ liệu & Độ phù hợp với thuật toán CHARM
- Mỗi hóa đơn (`InvoiceNo`) sau khi loại bỏ đơn hủy, số lượng/giá âm và các mã phí dịch vụ (`POST`, `DOT`, `M`, `C2`, `CRUK`, `AMAZONFEE`, `GIFT_*`, ...) được gom thành một giao dịch gồm tập các mã sản phẩm (`StockCode`, chuẩn hóa viết hoa) duy nhất.
- Dữ liệu bán lẻ trực tuyến có đặc tính **rất thưa (sparse)** (3.787 SKU, mật độ ma trận **0.691%**), chứa nhiều cụm sản phẩm mua kèm chặt chẽ như bộ nhãn thảo mộc **Herb Marker**, bộ đĩa trà **Regency Tea Plate**, đồ trang trí Giáng sinh bằng gỗ, và nhà búp bê **Poppy's Playhouse**.
- Thuật toán **Optimized CHARM** biểu diễn `Tidset` bằng mảng bit liền kề (`BitVector`, 309 từ 64-bit ≈ 2.47 KB/item) cùng phép giao `@inbounds @simd`, chỉ mất trung vị **74.16 ms** tại `minsup = 1.0%`. Bộ sinh luật (`src/rules.jl`) tra cứu chính xác support của mọi tập con qua `ClosureSupportIndex` và sinh toàn bộ các luật có hợp $A \cup B$ là tập đóng (trên dữ liệu này tại `minsup = 1.0%`, cả 2.026 tập phổ biến đều tự đóng nên tập luật thu được trùng khớp 100% với toàn bộ tập luật phổ biến).

#### Tải, làm sạch và chuyển đổi dữ liệu sang định dạng SPMF
Script `applications/market_basket/prepare_data.py` thực hiện:
1. Tải bộ dữ liệu **UCI Online Retail** (nếu chưa có sẵn cục bộ).
2. Loại bỏ bản ghi thiếu `InvoiceNo`/`StockCode`/`Description`, đơn hủy (`InvoiceNo` bắt đầu bằng `'C'`), `Quantity <= 0`, `UnitPrice <= 0`, và các mã phí dịch vụ không phải hàng hóa (`POST`, `DOT`, `M`, `BANK CHARGES`, ...).
3. Chuẩn hóa tên sản phẩm theo `StockCode`, gán `item_id` nguyên liên tục `1..M` tất định.
4. Gom theo `InvoiceNo` và xuất `data/applications/online_retail/transactions.txt` (chuẩn SPMF) cùng từ điển `item_mapping.csv`.

In [ ]:
import json
import pandas as pd

run([sys.executable, "applications/market_basket/prepare_data.py"])

with open("results/app1/dataset_summary.json", encoding="utf-8") as f:
    summary = json.load(f)

summary_rows = [
    ("Số dòng hóa đơn gốc (Raw rows)", f"{summary['raw_rows']:,}"),
    ("Loại bỏ thiếu InvoiceNo/StockCode/Description", f"{summary['dropped_missing_fields']:,}"),
    ("Loại bỏ đơn hàng bị hủy (InvoiceNo bắt đầu bằng 'C')", f"{summary['dropped_canceled_invoices']:,}"),
    ("Loại bỏ Quantity <= 0 hoặc UnitPrice <= 0", f"{summary['dropped_non_positive_qty_or_price']:,}"),
    ("Loại bỏ mã phí dịch vụ / không phải hàng hóa", f"{summary['dropped_non_product_items']:,}"),
    ("Số dòng sạch còn lại (Clean rows)", f"{summary['clean_rows']:,}"),
    ("Số giao dịch / hóa đơn (|D|)", f"{summary['n_transactions']:,}"),
    ("Số giao dịch có >= 2 sản phẩm", f"{summary['n_multi_item_transactions']:,}"),
    ("Số sản phẩm duy nhất (|I|)", f"{summary['n_unique_items']:,}"),
    ("Kích thước giỏ hàng (TB / Trung vị / Tối đa)", f"{summary['avg_basket_size']} / {summary['median_basket_size']} / {summary['max_basket_size']}"),
    ("Mật độ ma trận giao dịch", f"{summary['matrix_density'] * 100:.3f}%"),
]
display(pd.DataFrame(summary_rows, columns=["Chỉ số", "Giá trị"]))

#### Khai thác Tập phổ biến đóng & Sinh Luật kết hợp bằng `CharmMining.jl`
Toàn bộ kết quả được tạo trực tiếp bởi cài đặt **Optimized CHARM (`mine_file_bit`)** và module sinh luật kết hợp theo bao đóng (`generate_rules`, `top_rules_by_lift`) trong `src/rules.jl` — **không sử dụng bất kỳ thư viện FIM bên ngoài nào**.

In [ ]:
run(["julia", "--project=.", "applications/market_basket/run_mba.jl"])

#### Phân tích kết quả: Độ nhạy theo `minsup` & Top-10 Luật kết hợp theo Lift

In [ ]:
sens_df = pd.read_csv("results/app1/minsup_sensitivity.csv")
print("=== Bảng độ nhạy theo ngưỡng minsup ===")
display(sens_df)

top10_df = pd.read_csv("results/app1/top10_rules_lift_dedup.csv")
print("\n=== Top-10 Luật kết hợp theo Lift (minsup = 1.0%, minconf = 50%, đã khử cặp đối xứng) ===")
display(top10_df[["rank", "antecedent_names", "consequent_names", "support_count", "support_pct", "confidence_pct", "lift", "leverage"]])

#### Trực quan hóa kết quả

In [ ]:
from IPython.display import Image, display as ipy_display

run([sys.executable, "applications/market_basket/plot_mba.py"])

for fig_name in [
    "fig1_top15_products.png",
    "fig2_minsup_sensitivity.png",
    "fig3_rules_scatter.png",
    "fig4_top10_lift.png",
]:
    ipy_display(Image(filename=f"results/app1/figures/{fig_name}"))

#### Kiểm chứng hậu kiểm (Post-hoc Verification)
Tái lập toàn bộ các kiểm tra hậu kiểm: phân bố bộ 6 nhãn **Herb Marker** (gồm `CHIVES`), tỷ lệ nền mùa vụ tháng 10–12/2011 & **Lift tính riêng trong mùa**, và độ đa dạng `CustomerID`.

In [ ]:
run([sys.executable, "applications/market_basket/posthoc_analysis.py"])

with open("results/app1/posthoc_verification.json", encoding="utf-8") as f:
    ph = json.load(f)

print("=== 1. Phân bố số nhãn Herb Marker trong 289 hóa đơn có mua dòng này ===")
herb_dist_df = pd.DataFrame([
    {"Số nhãn Herb Marker trong giỏ": int(k), "Số hóa đơn": v,
     "Tỷ lệ trong 289 giỏ Herb (%)": round(100.0 * v / ph["herb_marker_analysis"]["baskets_with_at_least_1_herb"], 2)}
    for k, v in ph["herb_marker_analysis"]["size_distribution_among_herb_baskets"].items()
])
display(herb_dist_df)

print("\n=== 2. Kiểm chứng mức tập trung tháng 10–12/2011 (so với mốc nền sau ngày ra mắt 2011-06-09: 49.70%) ===")
season_df = pd.DataFrame([
    {"Sản phẩm": v["description"], "Ngày bán đầu tiên": v["first_invoice_date"],
     "Mốc nền 10–12/2011 sau ra mắt (%)": v["post_launch_baseline_oct_dec_2011_pct"],
     "HĐ 10–12/2011": v["oct_dec_2011_invoices"], "Tỷ lệ 10–12/2011 (%)": v["oct_dec_2011_pct"]}
    for v in ph["seasonality_analysis"]["wooden_decorations"].values()
])
display(season_df)
print("Lift trong giai đoạn 10–12/2011 (N=5,572):", ph["seasonality_analysis"]["in_season_oct_dec_2011_rules"])
print("Bộ 3 {TREE, STOCKING, SLEIGH}:", ph["seasonality_analysis"]["wooden_3_item_tree_stocking_sleigh"])

print("\n=== 3. Độ đa dạng khách hàng (CustomerID duy nhất) ===")
cust_df = pd.DataFrame([
    {"Cụm sản phẩm": k, "Tổng HĐ": v["invoices"], "HĐ có CustomerID": v["invoices_with_customer_id"], "Khách hàng duy nhất": v["unique_customers"]}
    for k, v in ph["customer_diversity"].items()
])
display(cust_df)

#### Diễn giải ý nghĩa kinh doanh & Khuyến nghị thực tiễn

Từ **962 luật kết hợp** khai thác bằng **Optimized CHARM (`CharmMining.jl`)** tại `minsup = 1.0%`, `minconf = 50%` và số liệu hậu kiểm (`posthoc_verification.json`), nhóm rút ra 4 nhận định nhất quán:

1. **Hành vi mua theo bộ sưu tập & bảo vệ biên lợi nhuận:** 6 luật có vế trái 2 nhãn trong Top-10 đạt xác suất mua thêm nhãn thứ 3 là **90.50% – 95.31%** (Lift `73.94 – 78.77`). Trong 289 giỏ mua dòng Herb Marker (6 SKU gồm cả `CHIVES`), **54.3% (`157/289`)** đã mua đủ cả 6 nhãn và **45.7% (`132/289`)** dừng ở 1–5 nhãn. Không nên giảm giá đại trà (gây giảm biên lợi nhuận); ưu tiên nút UX *"Thêm các nhãn còn thiếu"* và gợi ý nhắm riêng vào nhóm 45.7% giỏ chưa đủ bộ.
2. **Liên kết khác công năng trong cùng dòng thiết kế:** `REGENCY SUGAR BOWL GREEN => REGENCY MILK JUG PINK` (tần suất mua kèm lịch sử **75.5%**, Lift `45.36`) và `GARAGE KEY FOB => KEY FOB , SHED` (**71.9%**, Lift `38.24`) phù hợp hiển thị chéo trong mục *"Sản phẩm cùng bộ sưu tập"*.
3. **Cụm trang trí Giáng sinh tập trung ở mức ~1.5× vào tháng 10–12/2011 (một mùa quan sát được):** Cả 3 mã gỗ có ngày bán đầu tiên **2011-06-09**; tỷ lệ hóa đơn tháng 10–12/2011 đạt `74.06% – 76.25%` so với mốc nền sau ra mắt **49.70%** (tính trên toàn bộ hóa đơn cửa hàng từ 2011-06-09, tức gấp `1.49× – 1.53×`, còn ~24–26% diễn ra ở 06–09/2011). Lift tính riêng trong tháng 10–12/2011 là `18.64` và `16.29` (thấp hơn `46.33`/`42.08` do mẫu số 13 tháng gồm cả các tháng ngoài mùa và trước khi mã ra mắt). Bộ 3 `{TREE, STOCKING, SLEIGH}` đạt `159` HĐ (`0.80%`, dưới ngưỡng `1.0%`, `Conf({TREE, STOCKING} => SLEIGH) = 72.60%`). Nhất quán với mục 1: ưu tiên trưng bày theo chủ đề & gợi ý chéo thay vì giảm giá trực tiếp.
4. **Đồng bộ tồn kho & gợi ý chéo cặp đối xứng (*Dolly Girl* & *Spaceboy*):** `CHILDRENS CUTLERY DOLLY GIRL => SPACEBOY` (**76.2%**, Lift `40.97`, 228 HĐ trên **165 khách hàng duy nhất**) gợi ý hiển thị chéo hai mẫu và đồng bộ kế hoạch tồn kho thay vì suy diễn quá mức về nhân khẩu học.

Xem báo cáo chi tiết đầy đủ tại [`docs/app1_market_basket.md`](../docs/app1_market_basket.md).

### 3.2. Ứng dụng 2: Khai thác motif trên trình tự DNA (Human DNA — 7 họ protein)

**Tập dữ liệu:** `human.txt` (trình tự DNA người gán nhãn 7 họ protein, `CLASS_0`–`CLASS_6`).

**Biến đổi dữ liệu:** mỗi trình tự được mã hóa thành **tập các k-mer** (cửa sổ trượt, chỉ giữ k-mer gồm A/T/C/G) — một giao dịch
theo chuẩn SPMF. CHARM tìm các tập k-mer luôn xuất hiện cùng nhau (tập đóng).
- **Khai thác toàn cục:** `k = 6`, `minsup = 30%`, nhãn lớp `CLASS_i` được thêm vào giao dịch.
- **Khai thác theo từng họ protein:** `k = 8`, `minsup = 15%` số trình tự của lớp.

#### Tiền xử lý & thống kê mô tả
`eda.py`: chuẩn hóa chữ hoa, lọc độ dài ngoại lai (< 50 hoặc > 10.000 bp), loại chuỗi không có k-mer hợp lệ và khử trùng lặp.

In [ ]:
run([sys.executable, "applications/dna/eda.py"])

#### Khai thác tập đóng bằng Optimized CHARM
`run_dna.jl` dùng `to_vertical_bit` + `run_charm_bit` của `CharmMining.jl` (Chương 1).

In [ ]:
run(["julia", "--project=.", "applications/dna/run_dna.jl"])

#### Phân tích sinh học các pattern
`analysis.py` ghép các k-mer chồng lấp thành contig, tính GC%, số dinucleotide CpG, dịch mã 3 khung đọc và tỷ lệ amino acid kỵ nước.

In [ ]:
run([sys.executable, "applications/dna/analysis.py"])

uni = pd.read_csv("results/app2/bio_analysis/universal_motifs_bio_analysis.csv")
print(f"Số pattern phổ quát: {len(uni)}")
display(uni.head(15))

class_csv = Path("results/app2/bio_analysis/class_specific_bio_analysis.csv")
if class_csv.exists():
    display(pd.read_csv(class_csv).head(15))
else:
    print("Không có pattern đặc hiệu lớp đạt minsup toàn cục 30%.")

#### Pattern theo từng họ protein

In [ ]:
rows = []
for path in sorted(Path("results/app2/per_class_charm").glob("CLASS_*_charm_output.txt")):
    patterns = []
    for line in path.read_text().splitlines():
        items, sup = line.split(" #SUP: ")
        patterns.append((items.split(), int(sup)))
    longest = max(patterns, key=lambda p: (len(p[0]), p[1])) if patterns else ([], 0)
    rows.append({
        "class": path.name.split("_charm")[0],
        "closed_patterns": len(patterns),
        "max_kmers_in_pattern": len(longest[0]),
        "top_pattern_support": longest[1],
        "top_pattern": " + ".join(longest[0]),
    })
display(pd.DataFrame(rows))

Diễn giải chi tiết và các giới hạn của cách biểu diễn trình tự thành tập k-mer (mất thông tin thứ tự/vị trí) xem tại
[`docs/dna.md`](../docs/dna.md).

---
**Hoàn tất.** Toàn bộ kết quả được sinh lại trong `results/` (Chương 1, 3) và `comparison/` (Chương 2).